### Authentication and masks init

In [2]:
import ee


PROJECT_ID = 'mctnet-crop-classifier'

try:
    
    ee.Initialize(project=PROJECT_ID)
    print(f"✅ GEE initialisé avec succès sur le projet : {PROJECT_ID}")
except Exception as e:
    print("Ré-authentification nécessaire...")
    
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

#we define the regions of interest
states = ee.FeatureCollection("TIGER/2018/States")
california = states.filter(ee.Filter.eq('NAME', 'California')).geometry()
arkansas = states.filter(ee.Filter.eq('NAME', 'Arkansas')).geometry()

print("📍 Géométries de la Californie et de l'Arkansas chargées.")

✅ GEE initialisé avec succès sur le projet : mctnet-crop-classifier
📍 Géométries de la Californie et de l'Arkansas chargées.


In [3]:
#CDL mask 
cdl = ee.Image("USDA/NASS/CDL/2021")
label_layer = cdl.select('cropland')
confidence = cdl.select('confidence')

#ESA WorldCover mask to get only cropland areas
esa = ee.Image("ESA/WorldCover/v200/2021").select('Map')
agri_mask = esa.eq(40)

# We combine the CDL confidence mask with the ESA cropland mask to get a more reliable training set
training_mask = agri_mask.And(confidence.gt(95))
labeled_data = label_layer.updateMask(training_mask)

---

In [4]:
# import ee
# tasks = ee.data.listOperations()
# for t in tasks:
#     if t['metadata']['state'] in ['READY', 'RUNNING']:
#         ee.data.cancelOperation(t['name'])
# print("Toutes les tâches en cours ont été annulées. On repart de zéro.")

### Looping on different seeds across the states of California and Arkansas

In [ ]:
import ee
def mask_s2_clouds(image):
    #eleminate clouds and cirrus based on the QA60 band
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask)


def get_temporal_stack_multi(region, state_name, seed_val):
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                .filterBounds(region) \
                .filterDate('2021-01-01', '2021-12-31') \
                .map(mask_s2_clouds)
    
    image_list = []
    for i in range(36):
        start_day = i * 10 + 1
        composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                          .median() \
                          .select(band_names) \
                          .unmask(0)
        
        image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
    
    stack = ee.Image.cat(image_list)
    final_image = stack.addBands(labeled_data)

    
    samples = final_image.sample(
        region=region,
        scale=30,
        numPixels=5000, 
        seed=seed_val, 
        geometries=False,
        tileScale=16,
        dropNulls=True
    )

   
    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=f'MCTNet_Final_{seed_val}_{state_name}', 
        fileFormat='CSV'
    )
    task.start()
    return f"✅ Tâche lancée : {state_name} avec Seed {seed_val}"

In [ ]:
debut_seed = 201
fin_seed = 230
for s in range(debut_seed, fin_seed + 1):
    print(get_temporal_stack_multi(california, "California", s))


✅ Tâche lancée : California avec Seed 201
✅ Tâche lancée : California avec Seed 202
✅ Tâche lancée : California avec Seed 203
✅ Tâche lancée : California avec Seed 204
✅ Tâche lancée : California avec Seed 205
✅ Tâche lancée : California avec Seed 206
✅ Tâche lancée : California avec Seed 207
✅ Tâche lancée : California avec Seed 208
✅ Tâche lancée : California avec Seed 209
✅ Tâche lancée : California avec Seed 210
✅ Tâche lancée : California avec Seed 211
✅ Tâche lancée : California avec Seed 212
✅ Tâche lancée : California avec Seed 213
✅ Tâche lancée : California avec Seed 214
✅ Tâche lancée : California avec Seed 215
✅ Tâche lancée : California avec Seed 216
✅ Tâche lancée : California avec Seed 217
✅ Tâche lancée : California avec Seed 218
✅ Tâche lancée : California avec Seed 219
✅ Tâche lancée : California avec Seed 220
✅ Tâche lancée : California avec Seed 221
✅ Tâche lancée : California avec Seed 222
✅ Tâche lancée : California avec Seed 223
✅ Tâche lancée : California avec S

In [ ]:
debut_seed = 101
fin_seed = 120
for s in range(debut_seed, fin_seed + 1):
    print(get_temporal_stack_multi(arkansas, "Arkansas", s))

✅ Tâche lancée : Arkansas avec Seed 101
✅ Tâche lancée : Arkansas avec Seed 102
✅ Tâche lancée : Arkansas avec Seed 103
✅ Tâche lancée : Arkansas avec Seed 104
✅ Tâche lancée : Arkansas avec Seed 105
✅ Tâche lancée : Arkansas avec Seed 106
✅ Tâche lancée : Arkansas avec Seed 107
✅ Tâche lancée : Arkansas avec Seed 108
✅ Tâche lancée : Arkansas avec Seed 109
✅ Tâche lancée : Arkansas avec Seed 110
✅ Tâche lancée : Arkansas avec Seed 111
✅ Tâche lancée : Arkansas avec Seed 112
✅ Tâche lancée : Arkansas avec Seed 113
✅ Tâche lancée : Arkansas avec Seed 114
✅ Tâche lancée : Arkansas avec Seed 115
✅ Tâche lancée : Arkansas avec Seed 116
✅ Tâche lancée : Arkansas avec Seed 117
✅ Tâche lancée : Arkansas avec Seed 118
✅ Tâche lancée : Arkansas avec Seed 119
✅ Tâche lancée : Arkansas avec Seed 120


#### Strategy Note:

* This strategy was to launch multiple export tasks with different seeds to get a more diverse training set, while avoiding the risk of overloading GEE with too many simultaneous tasks.
* But the downside is some crops might be underrepresented in some exports. 
**Example:** For example pistachios are very few compared to other crops like corn; so the change to get a pixels among the thousands other pixels is really low.

--- 

### Reduce the search area; focus on regions specific to certain crops:

In [ ]:
import ee
# we verify if we got enough pixels in the new rectangles:

pistachio_region = ee.Geometry.Rectangle([-120.5, 35.0, -118.5, 36.5])
arkansas_delta = ee.Geometry.Rectangle([-91.8, 33.5, -90.5, 36.0])


def count_pixels(region, class_code, label_name):
    count = labeled_data.eq(class_code).selfMask().reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=region,
        scale=30,
        maxPixels=1e9
    ).get('cropland') 
    
    return f"📍 {label_name} (Code {class_code}): {count.getInfo()} pixels trouvés"


print("--- VÉRIFICATION CALIFORNIE ---")
print(count_pixels(pistachio_region, 76, "Pistaches"))

print("\n--- VÉRIFICATION ARKANSAS ---")
print(count_pixels(arkansas_delta, 1, "Soybeans"))
print(count_pixels(arkansas_delta, 3, "Rice"))

--- VÉRIFICATION CALIFORNIE ---
📍 Pistaches (Code 76): 32352 pixels trouvés

--- VÉRIFICATION ARKANSAS ---
📍 Soybeans (Code 1): 1405225 pixels trouvés
📍 Rice (Code 3): 2206108 pixels trouvés


In [ ]:
import ee
#in this strategy we first filter the labeled data by crop type to isolate the target class, 
#then we sample spectral data from the temporal stack. 
california_valley = ee.Geometry.Rectangle([-120.5, 35.0, -118.5, 36.5])
arkansas_delta = ee.Geometry.Rectangle([-91.8, 33.5, -90.5, 36.0])


targets = {
    'Arkansas': {
        'region': arkansas_delta,
        'crops': {
            1: {'name': 'Soybeans', 'num': 4400}, 
            3: {'name': 'Rice', 'num': 2200},
            2: {'name': 'Cotton', 'num': 500},
            5: {'name': 'Corn', 'num': 1300}, 
            176: {'name': 'Others_Grass', 'num': 400}
        }
    },
    'California': {
        'region': california_valley,
        'crops': {
            69: {'name': 'Grapes', 'num': 1800},
            3:  {'name': 'Rice', 'num': 1800},
            36: {'name': 'Alfalfa', 'num': 700},
            75: {'name': 'Almonds', 'num': 500},
            76: {'name': 'Pistachios', 'num': 400},
            176: {'name': 'Others_Grass', 'num': 3300}
        }
    }
}

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def run_extraction_ultime():
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    for state, config in targets.items():
        region = config['region']
        print(f"--- 🛰️ Préparation du Stack pour {state} ---")
        
        
        s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                    .filterBounds(region) \
                    .filterDate('2021-01-01', '2021-12-31') \
                    .map(mask_s2_clouds)

        image_list = []
        for i in range(36):
            start_day = i * 10 + 1
            composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                              .median() \
                              .select(band_names) \
                              .unmask(0) 
            image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
        
        stack = ee.Image.cat(image_list)

        for code, info in config['crops'].items():
            crop_name = info['name']
            num_pix = info['num']
            
         
            
            points = labeled_data.eq(code).selfMask().stratifiedSample(
                numPoints=num_pix,
                classBand='cropland',
                region=region,
                scale=30,
                geometries=True,
                seed=42,
                dropNulls=True
            )

            
            final_samples = stack.sampleRegions(
                collection=points,
                properties=['cropland'],
                scale=30,
                tileScale=16
            )

            
            desc = f"MCTNet_{state}_{crop_name}_{num_pix}"
            task = ee.batch.Export.table.toDrive(
                collection=final_samples,
                description=desc,
                fileFormat='CSV'
            )
            task.start()
            print(f"✅ Tâche lancée : {desc}")


run_extraction_ultime()

--- 🛰️ Préparation du Stack pour Arkansas ---
✅ Tâche lancée : MCTNet_Arkansas_Soybeans_4400
✅ Tâche lancée : MCTNet_Arkansas_Rice_2200
✅ Tâche lancée : MCTNet_Arkansas_Cotton_500
✅ Tâche lancée : MCTNet_Arkansas_Corn_1300
✅ Tâche lancée : MCTNet_Arkansas_Others_Grass_400
--- 🛰️ Préparation du Stack pour California ---
✅ Tâche lancée : MCTNet_California_Grapes_1800
✅ Tâche lancée : MCTNet_California_Rice_1800
✅ Tâche lancée : MCTNet_California_Alfalfa_700
✅ Tâche lancée : MCTNet_California_Almonds_500
✅ Tâche lancée : MCTNet_California_Pistachios_400
✅ Tâche lancée : MCTNet_California_Others_Grass_3300


In [ ]:
import ee
# we keep the same strategy but we target only the missing crops with a larger region to maximize the chances of finding enough pixels.

ca_patch_region = ee.Geometry.Rectangle([-121.5, 34.5, -118.0, 37.5])
ar_patch_region = ee.Geometry.Rectangle([-92.5, 33.0, -90.0, 36.5])


patch_targets = {
    'California': {
        'region': ca_patch_region,
        'crops': {
            69: {'name': 'Grapes_Patch', 'num': 1200},
            3:  {'name': 'Rice_Patch', 'num': 1300},
            36: {'name': 'Alfalfa_Patch', 'num': 650},
            75: {'name': 'Almonds_Patch', 'num': 550},
            76: {'name': 'Pistachios_Patch', 'num': 600}
        }
    },
    'Arkansas': {
        'region': ar_patch_region,
        'crops': {
            1:   {'name': 'Soybeans_Patch', 'num': 1800},
            3:   {'name': 'Rice_Patch', 'num': 1400},
            2:   {'name': 'Cotton_Patch', 'num': 450},
            176: {'name': 'Others_Patch', 'num': 400} 
        }
    }
}

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def run_patch_extraction():
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    for state, config in patch_targets.items():
        region = config['region']
        print(f"--- 🩹 Patching {state} ---")
        

        s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                    .filterBounds(region) \
                    .filterDate('2021-01-01', '2021-12-31') \
                    .map(mask_s2_clouds)

        image_list = []
        for i in range(36):
            start_day = i * 10 + 1
            composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                              .median() \
                              .select(band_names) \
                              .unmask(0)
            image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
        
        stack = ee.Image.cat(image_list)

        for code, info in config['crops'].items():
            crop_name = info['name']
            num_pix = info['num']
            
          
            points = labeled_data.eq(code).selfMask().stratifiedSample(
                numPoints=num_pix,
                classBand='cropland',
                region=region,
                scale=30,
                geometries=True, 
                seed=999,        
                dropNulls=True
            )

            final_samples = stack.sampleRegions(
                collection=points,
                properties=['cropland'],
                scale=30,
                tileScale=16
            )

            desc = f"PATCH_{state}_{crop_name}"
            ee.batch.Export.table.toDrive(
                collection=final_samples,
                description=desc,
                fileFormat='CSV'
            ).start()
            print(f"🚀 Lancé : {desc}")

run_patch_extraction()

--- 🩹 Patching California ---
🚀 Lancé : PATCH_California_Grapes_Patch
🚀 Lancé : PATCH_California_Rice_Patch
🚀 Lancé : PATCH_California_Alfalfa_Patch
🚀 Lancé : PATCH_California_Almonds_Patch
🚀 Lancé : PATCH_California_Pistachios_Patch
--- 🩹 Patching Arkansas ---
🚀 Lancé : PATCH_Arkansas_Soybeans_Patch
🚀 Lancé : PATCH_Arkansas_Rice_Patch
🚀 Lancé : PATCH_Arkansas_Cotton_Patch
🚀 Lancé : PATCH_Arkansas_Others_Patch


#### Strategy Note:

* This strategy focuses on launching export tasks by first filtering the labeled data by crop type (`eq(code).selfMask()`) to isolate the target class, then sampling the spectral data.
* Even with this isolation, using a single seed (like `seed=42`) can sometimes result in **empty files**. This happens because, in specific regions, the random distribution of that single seed might not land on any valid pixels for very rare crops.


**The "Mixed" Solution:** To overcome this, we move to a **Mixed Strategy**: combining the **label isolation** (to force focus) with **multiple seed iterations**. This ensures that if one seed misses the rare pixels, the subsequent seeds will capture them, guaranteeing a complete and diverse training set without empty files.

---

### the most effective strategy is the mixed (changing seeds and filtering by the crop)

In [ ]:
import ee
#this is by far the best way 

# MIXED STRATEGY:
# 1. MULTI-SEED: Iterates 5 random seeds to maximize pixel diversity and capture rare samples.
# 2. TARGETED EXTRACTION: Masks by specific crop IDs to extract only required classes 
# this will help to have a more balanced dataset and avoid the dominance of certain crops

arkansas_delta = ee.Geometry.Rectangle([-91.8, 33.5, -90.5, 36.0])
california_valley = ee.Geometry.Rectangle([-120.5, 35.0, -118.5, 36.5])


targets = {
    'Arkansas': {
        'region': arkansas_delta,
        'crops': {
            1: {'name': 'Soybeans', 'num': 4400}, 
            3: {'name': 'Rice', 'num': 2200},
            2: {'name': 'Cotton', 'num': 500},
            5: {'name': 'Corn', 'num': 1300},
            176: {'name': 'Others_Grass', 'num': 400}
        }
    },
    'California': {
        'region': california_valley,
        'crops': {
            69: {'name': 'Grapes', 'num': 1800},
            3:  {'name': 'Rice', 'num': 1800},
            36: {'name': 'Alfalfa', 'num': 700},
            75: {'name': 'Almonds', 'num': 500},
            76: {'name': 'Pistachios', 'num': 400},
            176: {'name': 'Others_Grass', 'num': 3300}
        }
    }
}

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

def run_multi_seed_extraction():
    #5 different seeds
    seeds = [42, 123, 777, 2024, 999]
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    for seed_val in seeds:
        print(f"\n🌟 --- DEBUT DE LA SEED: {seed_val} --- 🌟")
        
        for state, config in targets.items():
            region = config['region']
            
            
            s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                        .filterBounds(region) \
                        .filterDate('2021-01-01', '2021-12-31') \
                        .map(mask_s2_clouds)

            image_list = []
            for i in range(36):
                start_day = i * 10 + 1
                composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                                  .median() \
                                  .select(band_names) \
                                  .unmask(0)
                image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
            
            stack = ee.Image.cat(image_list)

            for code, info in config['crops'].items():
                crop_name = info['name']
                num_pix = info['num']
                
                
                points = labeled_data.eq(code).selfMask().stratifiedSample(
                    numPoints=num_pix,
                    classBand='cropland',
                    region=region,
                    scale=30,
                    geometries=True,
                    seed=seed_val, 
                    dropNulls=True
                )

                samples = stack.sampleRegions(
                    collection=points,
                    properties=['cropland'],
                    scale=30,
                    tileScale=16
                )

                
                desc = f"MCTNet_{state}_{crop_name}_Seed{seed_val}"
                task = ee.batch.Export.table.toDrive(
                    collection=samples,
                    description=desc,
                    fileFormat='CSV'
                )
                task.start()
                print(f"✅ Lancé : {desc}")

run_multi_seed_extraction()


🌟 --- DEBUT DE LA SEED: 42 --- 🌟
✅ Lancé : MCTNet_Arkansas_Soybeans_Seed42
✅ Lancé : MCTNet_Arkansas_Rice_Seed42
✅ Lancé : MCTNet_Arkansas_Cotton_Seed42
✅ Lancé : MCTNet_Arkansas_Corn_Seed42
✅ Lancé : MCTNet_Arkansas_Others_Grass_Seed42
✅ Lancé : MCTNet_California_Grapes_Seed42
✅ Lancé : MCTNet_California_Rice_Seed42
✅ Lancé : MCTNet_California_Alfalfa_Seed42
✅ Lancé : MCTNet_California_Almonds_Seed42
✅ Lancé : MCTNet_California_Pistachios_Seed42
✅ Lancé : MCTNet_California_Others_Grass_Seed42

🌟 --- DEBUT DE LA SEED: 123 --- 🌟
✅ Lancé : MCTNet_Arkansas_Soybeans_Seed123
✅ Lancé : MCTNet_Arkansas_Rice_Seed123
✅ Lancé : MCTNet_Arkansas_Cotton_Seed123
✅ Lancé : MCTNet_Arkansas_Corn_Seed123
✅ Lancé : MCTNet_Arkansas_Others_Grass_Seed123
✅ Lancé : MCTNet_California_Grapes_Seed123
✅ Lancé : MCTNet_California_Rice_Seed123
✅ Lancé : MCTNet_California_Alfalfa_Seed123
✅ Lancé : MCTNet_California_Almonds_Seed123
✅ Lancé : MCTNet_California_Pistachios_Seed123
✅ Lancé : MCTNet_California_Others_Gr

---

## PART 2:

In [3]:
import ee

# Initialisation des géométries
arkansas_delta = ee.Geometry.Rectangle([-91.8, 33.5, -90.5, 36.0])
california_valley = ee.Geometry.Rectangle([-120.5, 35.0, -118.5, 36.5])

targets = {
    'Arkansas': {
        'region': arkansas_delta,
        'crops': {
            1: {'name': 'Soybeans', 'num': 5000}, 
            3: {'name': 'Rice', 'num': 5000},
            2: {'name': 'Cotton', 'num': 5000},
            5: {'name': 'Corn', 'num': 5000},
            176: {'name': 'Others_Grass', 'num': 5000}
        }
    },
    'California': {
        'region': california_valley,
        'crops': {
            69: {'name': 'Grapes', 'num': 5000},
            3:  {'name': 'Rice', 'num': 5000},
            36: {'name': 'Alfalfa', 'num': 5000},
            75: {'name': 'Almonds', 'num': 5000},
            76: {'name': 'Pistachios', 'num': 5000},
            176: {'name': 'Others_Grass', 'num': 5000}
        }
    }
}

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

# --- NOUVEAU : FONCTION POUR LES COVARIABLES ---
def get_environmental_covariates(region):
    #Topo
    topo = ee.Image("USGS/SRTMGL1_003").clip(region)
    elevation = topo.select('elevation').rename('topo_elevation')
    slope = ee.Terrain.slope(elevation).rename('topo_slope')

    # Sol
    soil = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02") \
             .select('b0').rename('soil_texture') \
             .resample('bilinear').reproject(crs='EPSG:4326', scale=30)
    
    #Climat 
    climate = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
                .filterDate('2021-01-01', '2021-12-31') \
                .select(['temperature_2m', 'total_precipitation_sum']) \
                .mean() \
                .rename(['clim_temp', 'clim_precip']) \
                .resample('bilinear').reproject(crs='EPSG:4326', scale=30)
    
    return elevation.addBands([slope, soil, climate]).unmask(0)

def run_multi_seed_extraction():
    seeds = [42, 123, 777, 2024, 999]
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    
    labeled_data = ee.Image("USDA/NASS/CDL/2021").select('cropland')
    
    for seed_val in seeds:
        print(f"\n🌟 --- DEBUT DE LA SEED: {seed_val} --- 🌟")
        
        for state, config in targets.items():
            region = config['region']
            
            # --- INTEGRATION DES COVARIABLES ---
            covars = get_environmental_covariates(region)
            
            s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                        .filterBounds(region) \
                        .filterDate('2021-01-01', '2021-12-31') \
                        .map(mask_s2_clouds)

            image_list = []
            for i in range(36):
                start_day = i * 10 + 1
                composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                                  .median() \
                                  .select(band_names) \
                                  .unmask(0)
                image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
            
            # STACK FINAL : Sentinel-2 + Covariables
            stack = ee.Image.cat(image_list).addBands(covars)

            for code, info in config['crops'].items():
                crop_name = info['name']
                num_pix = info['num']
                
                points = labeled_data.eq(code).selfMask().stratifiedSample(
                    numPoints=num_pix,
                    classBand='cropland',
                    region=region,
                    scale=30,
                    geometries=True,
                    seed=seed_val, 
                    dropNulls=True
                )

                # Échantillonnage sur le stack augmenté
                samples = stack.sampleRegions(
                    collection=points,
                    properties=['cropland'],
                    scale=30,
                    tileScale=16
                )

                desc = f"MCTNet_P2_{state}_{crop_name}_Seed{seed_val}"
                task = ee.batch.Export.table.toDrive(
                    collection=samples,
                    description=desc,
                    fileFormat='CSV'
                )
                task.start()
                print(f"✅ Lancé (Partie 2) : {desc}")

# Lancer l'extraction
run_multi_seed_extraction()


🌟 --- DEBUT DE LA SEED: 42 --- 🌟
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Soybeans_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Rice_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Cotton_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Corn_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Others_Grass_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Grapes_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Rice_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Alfalfa_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Almonds_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Pistachios_Seed42
✅ Lancé (Partie 2) : MCTNet_P2_California_Others_Grass_Seed42

🌟 --- DEBUT DE LA SEED: 123 --- 🌟
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Soybeans_Seed123
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Rice_Seed123
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Cotton_Seed123
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Corn_Seed123
✅ Lancé (Partie 2) : MCTNet_P2_Arkansas_Others_Grass_Seed123
✅ Lancé (Partie 2) : MCTNet_P2_C

In [ ]:
import ee

arkansas_secure = ee.Geometry.Rectangle([-91.5, 35.0, -90.0, 36.5])

# California Sacramento Valley (Le temple du Rice)
# On se concentre sur la zone entre Williams et Chico
california_secure = ee.Geometry.Rectangle([-122.4, 38.8, -121.4, 40.2])


targets = {
    'Arkansas': {
        'region': arkansas_delta,
        'crops': {
            5: {'name': 'Corn', 'num': 1200},
        }
    },
}

def mask_s2_clouds(image):
    qa = image.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.updateMask(mask)

# --- NOUVEAU : FONCTION POUR LES COVARIABLES ---
def get_environmental_covariates(region):
    #Topo
    topo = ee.Image("USGS/SRTMGL1_003").clip(region)
    elevation = topo.select('elevation').rename('topo_elevation')
    slope = ee.Terrain.slope(elevation).rename('topo_slope')

    # Sol
    soil = ee.Image("OpenLandMap/SOL/SOL_TEXTURE-CLASS_USDA-TT_M/v02") \
             .select('b0').rename('soil_texture') \
             .resample('bilinear').reproject(crs='EPSG:4326', scale=30)
    
    #Climat 
    climate = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
                .filterDate('2021-01-01', '2021-12-31') \
                .select(['temperature_2m', 'total_precipitation_sum']) \
                .mean() \
                .rename(['clim_temp', 'clim_precip']) \
                .resample('bilinear').reproject(crs='EPSG:4326', scale=30)
    
    return elevation.addBands([slope, soil, climate]).unmask(0)
def run_multi_seed_extraction():
    seeds = [42, 123, 777, 2024, 999]
    #seeds = [ 1337, 888, 555, 101, 314]
    band_names = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    
    
    labeled_data = ee.Image("USDA/NASS/CDL/2021").select('cropland')
    
    for seed_val in seeds:
        print(f"\n🌟 --- DEBUT DE LA SEED: {seed_val} --- 🌟")
        
        for state, config in targets.items():
            region = config['region']
            
            # --- INTEGRATION DES COVARIABLES ---
            covars = get_environmental_covariates(region)
            
            s2_col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
                        .filterBounds(region) \
                        .filterDate('2021-01-01', '2021-12-31') \
                        .map(mask_s2_clouds)

            image_list = []
            for i in range(36):
                start_day = i * 10 + 1
                composite = s2_col.filter(ee.Filter.dayOfYear(start_day, start_day + 10)) \
                                  .median() \
                                  .select(band_names) \
                                  .unmask(0)
                image_list.append(composite.rename([f'd{i}_{b}' for b in band_names]))
            
            # STACK FINAL : Sentinel-2 + Covariables
            stack = ee.Image.cat(image_list).addBands(covars)

            for code, info in config['crops'].items():
                crop_name = info['name']
                num_pix = info['num']
                
                points = labeled_data.eq(code).selfMask().stratifiedSample(
                    numPoints=num_pix,
                    classBand='cropland',
                    region=region,
                    scale=30,
                    geometries=True,
                    seed=seed_val, 
                    dropNulls=True
                )

                # Échantillonnage sur le stack augmenté
                samples = stack.sampleRegions(
                    collection=points,
                    properties=['cropland'],
                    scale=30,
                    tileScale=16
                )

                desc = f"MCTNet_{state}_{crop_name}_Seed{seed_val}"
                task = ee.batch.Export.table.toDrive(
                    collection=samples,
                    description=desc,
                    fileFormat='CSV'
                )
                task.start()
                print(f"✅ Lancé (Partie 2) : {desc}")

# Lancer l'extraction
run_multi_seed_extraction()


🌟 --- DEBUT DE LA SEED: 42 --- 🌟
✅ Lancé (Partie 2) : MCTNet_Arkansas_Corn_Seed42

🌟 --- DEBUT DE LA SEED: 123 --- 🌟
✅ Lancé (Partie 2) : MCTNet_Arkansas_Corn_Seed123

🌟 --- DEBUT DE LA SEED: 777 --- 🌟
✅ Lancé (Partie 2) : MCTNet_Arkansas_Corn_Seed777

🌟 --- DEBUT DE LA SEED: 2024 --- 🌟
✅ Lancé (Partie 2) : MCTNet_Arkansas_Corn_Seed2024

🌟 --- DEBUT DE LA SEED: 999 --- 🌟
✅ Lancé (Partie 2) : MCTNet_Arkansas_Corn_Seed999
